# lipshift, Step 0: go / no-go

Purpose: check, in about 30 minutes, that everything the project depends on works on Colab.

1. GPU and Python version
2. Auto-AVSR code and mediapipe mouth cropping
3. MAVOS-DD access, file layout and metadata counts per language and generator
4. English VSR checkpoint loads; features extract from real MAVOS-DD clips on GPU
5. (Optional) Spanish VSR checkpoint loads into the same encoder, which decides Study A

Before running: Runtime > Change runtime type > T4 GPU.
If an earlier version of this notebook already ran on your runtime, first do
Runtime > Disconnect and delete runtime (it downgraded protobuf; a fresh VM undoes that).
Run cell 1, then Runtime > Restart session, then run cell 2 onward.
Results are written to Google Drive under `MyDrive/lipshift/step0/`.

Licenses: Auto-AVSR code is Apache-2.0 (pinned commit); the BlazeFace model is from the official MediaPipe model bucket; its pretrained models are for non-commercial
benchmarking use. MAVOS-DD is CC BY-NC-SA 4.0. Nothing downloaded here goes into the git repo.

In [ ]:
# Cell 1: installs. Then Runtime > Restart session.
# Auto-AVSR's own mouth cropper uses `mediapipe.solutions`, which no longer exists in current
# mediapipe. We use the MediaPipe Tasks FaceDetector instead (see step0_helpers.py).
# mediapipe 1.2.2 has no protobuf requirement, so Colab's Google packages are not touched.
!git clone -q https://github.com/mpc001/auto_avsr /content/auto_avsr || true
!git -C /content/auto_avsr checkout -q 182b62837773ab01052d4ac21ef1d2203ea7d267
!pip install -q mediapipe==1.2.2 sentencepiece scikit-image
# BlazeFace short-range model (official MediaPipe model bucket, versioned path)
!wget -q -nc https://storage.googleapis.com/mediapipe-models/face_detector/blaze_face_short_range/float16/1/blaze_face_short_range.tflite -O /content/blaze_face_short_range.tflite
!ls -l /content/blaze_face_short_range.tflite
print("Done. Now: Runtime > Restart session, then run Cell 2.")

In [ ]:
# Cell 2: environment report
import json, platform, sys, importlib
from pathlib import Path
report = {"python": platform.python_version()}
for mod in ["torch", "torchaudio", "torchvision", "numpy", "cv2", "mediapipe", "huggingface_hub", "datasets"]:
    try:
        report[mod] = importlib.import_module(mod).__version__
    except Exception as e:  # noqa: BLE001
        report[mod] = f"MISSING ({type(e).__name__})"
import torch
report["cuda"] = torch.cuda.is_available()
report["gpu"] = torch.cuda.get_device_name(0) if report["cuda"] else None
import cv2
from mediapipe.tasks.python import vision  # noqa: F401  (Tasks API present)
FACE_MODEL = "/content/blaze_face_short_range.tflite"
report["cv2_has_VideoCapture"] = hasattr(cv2, "VideoCapture")
report["face_model_exists"] = Path(FACE_MODEL).exists()
print(json.dumps(report, indent=2))
assert report["cuda"], "No GPU: Runtime > Change runtime type > T4 GPU"
assert report["cv2_has_VideoCapture"], "Broken cv2: !pip install --force-reinstall opencv-contrib-python, restart"
assert report["face_model_exists"], "Face model missing: re-run the wget line in Cell 1"
assert not str(report["torchaudio"]).startswith("MISSING"), "Run: !pip install torchaudio (matching Colab's torch)"

In [ ]:
# Cell 3: Google Drive output folders
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")
WORK = Path("/content/drive/MyDrive/lipshift")
OUT = WORK / "step0"
WEIGHTS = WORK / "weights"
CLIPS = Path("/content/clips")      # local disk, not Drive: videos are not kept
for p in (OUT, WEIGHTS, CLIPS):
    p.mkdir(parents=True, exist_ok=True)
AUTO_AVSR = "/content/auto_avsr"
(OUT / "env.json").write_text(json.dumps(report, indent=2))
print("Writing to", OUT)

In [ ]:
%%writefile /content/step0_helpers.py
"""Step 0 helpers: video loading, mouth cropping and VSR feature extraction.

Wraps the official Auto-AVSR code (https://github.com/mpc001/auto_avsr,
Apache-2.0) with two fixes for current library versions:

1. Video is read with OpenCV, because ``torchvision.io.read_video`` is not
   present in torchvision 0.29.
2. Face keypoints come from the MediaPipe Tasks ``FaceDetector``, because the
   legacy ``mediapipe.solutions`` API used by Auto-AVSR is absent from mediapipe
   0.10.30+ and 1.x. Requires a BlazeFace short-range ``.tflite`` model file.

Pretrained weights are not covered by the Apache-2.0 code license: see the
model zoo terms (non-commercial, benchmarking use).
"""

from __future__ import annotations

import logging
import sys
import time
from pathlib import Path

import cv2
import numpy as np
import torch

logger = logging.getLogger(__name__)


def load_video_rgb(
    path: str | Path, max_frames: int | None = None
) -> tuple[np.ndarray, float]:
    """Read a video file into an RGB uint8 array of shape (T, H, W, 3).

    Args:
        path: Video file path.
        max_frames: Stop after this many frames (None reads all).

    Returns:
        Frames and the frame rate reported by the container.
    """
    cap = cv2.VideoCapture(str(path))
    if not cap.isOpened():
        raise OSError(f"Cannot open video: {path}")
    fps = float(cap.get(cv2.CAP_PROP_FPS)) or 25.0
    frames: list[np.ndarray] = []
    while True:
        ok, frame_bgr = cap.read()
        if not ok:
            break
        frames.append(cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB))
        if max_frames is not None and len(frames) >= max_frames:
            break
    cap.release()
    if not frames:
        raise ValueError(f"No frames decoded from: {path}")
    return np.stack(frames), fps


def add_auto_avsr_to_path(auto_avsr_dir: str | Path) -> None:
    """Make the cloned Auto-AVSR repository importable."""
    p = str(Path(auto_avsr_dir).resolve())
    if p not in sys.path:
        sys.path.insert(0, p)


class TasksLandmarksDetector:
    """Drop-in replacement for Auto-AVSR's mediapipe ``LandmarksDetector``.

    Uses the MediaPipe Tasks ``FaceDetector`` (mediapipe 0.10.30+ and 1.x), which
    replaced the removed ``mediapipe.solutions`` API. BlazeFace returns six
    keypoints in the same order as the legacy API (right eye, left eye, nose tip,
    mouth center, right ear, left ear); Auto-AVSR uses the first four.

    Differences from the original: only the short-range BlazeFace model is used
    (the legacy code tried full range first), and the largest face is selected
    by bounding-box area.
    """

    def __init__(self, model_path: str | Path, min_confidence: float = 0.5) -> None:
        from mediapipe.tasks import python as mp_tasks
        from mediapipe.tasks.python import vision

        options = vision.FaceDetectorOptions(
            base_options=mp_tasks.BaseOptions(model_asset_path=str(model_path)),
            min_detection_confidence=min_confidence,
        )
        self._detector = vision.FaceDetector.create_from_options(options)

    def close(self) -> None:
        """Release the detector explicitly (avoids noisy errors at interpreter exit)."""
        if self._detector is not None:
            self._detector.close()
            self._detector = None

    def __call__(self, video_frames: np.ndarray) -> list[np.ndarray | None]:
        import mediapipe as mp

        landmarks: list[np.ndarray | None] = []
        for frame in video_frames:
            h, w = frame.shape[:2]
            image = mp.Image(
                image_format=mp.ImageFormat.SRGB, data=np.ascontiguousarray(frame)
            )
            detections = self._detector.detect(image).detections
            if not detections:
                landmarks.append(None)
                continue
            best = max(
                detections, key=lambda d: d.bounding_box.width * d.bounding_box.height
            )
            pts = [[int(k.x * w), int(k.y * h)] for k in best.keypoints[:4]]
            landmarks.append(np.array(pts))
        if all(lm is None for lm in landmarks):
            raise AssertionError("Cannot detect any frames in the video")
        return landmarks


def crop_mouths(
    frames_rgb: np.ndarray, auto_avsr_dir: str | Path, face_model_path: str | Path
) -> np.ndarray:
    """Detect the face per frame and return aligned mouth crops (T, 96, 96, 3).

    Raises:
        AssertionError: if no face is found in any frame.
        RuntimeError: if cropping returns nothing.
    """
    add_auto_avsr_to_path(auto_avsr_dir)
    from preparation.detectors.mediapipe.video_process import VideoProcess

    detector = TasksLandmarksDetector(face_model_path)
    try:
        landmarks = detector(frames_rgb)
    finally:
        detector.close()
    mouths = VideoProcess(convert_gray=False)(frames_rgb, landmarks)
    if mouths is None:
        raise RuntimeError("Mouth cropping returned None (no usable landmarks).")
    return np.asarray(mouths)


class VSREncoder(torch.nn.Module):
    """Auto-AVSR visual front-end + conformer encoder, without the decoder."""

    def __init__(self, auto_avsr_dir: str | Path) -> None:
        super().__init__()
        add_auto_avsr_to_path(auto_avsr_dir)
        from datamodule.transforms import TextTransform
        from espnet.nets.pytorch_backend.e2e_asr_conformer import E2E

        n_tokens = len(TextTransform().token_list)
        self.model = E2E(n_tokens, "video", ctc_weight=0.1)

    def load_checkpoint(self, ckpt_path: str | Path) -> dict[str, list[str]]:
        """Load weights non-strictly and report key mismatches."""
        state = torch.load(str(ckpt_path), map_location="cpu")
        if isinstance(state, dict) and "state_dict" in state:
            state = state["state_dict"]
        result = self.model.load_state_dict(state, strict=False)
        return {
            "missing": list(result.missing_keys),
            "unexpected": list(result.unexpected_keys),
        }

    @torch.inference_mode()
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Map (T, 1, 88, 88) normalized mouth frames to (T, D) features."""
        y = self.model.frontend(x.unsqueeze(0))
        y = self.model.proj_encoder(y)
        y, _ = self.model.encoder(y, None)
        return y.squeeze(0)


def preprocess_mouths(
    mouths_rgb: np.ndarray, auto_avsr_dir: str | Path
) -> torch.Tensor:
    """Apply the official test-time transform: /255, center crop 88, gray, normalize."""
    add_auto_avsr_to_path(auto_avsr_dir)
    from datamodule.transforms import VideoTransform

    video = torch.from_numpy(mouths_rgb).permute(0, 3, 1, 2)  # (T, C, H, W)
    return VideoTransform("test")(video)


def extract_features(
    video_path: str | Path,
    encoder: VSREncoder,
    auto_avsr_dir: str | Path,
    face_model_path: str | Path,
    device: str = "cpu",
    max_frames: int | None = 250,
) -> tuple[np.ndarray, dict[str, float]]:
    """Full path: video file -> mouth crops -> (T, D) float16 features and timings."""
    t0 = time.perf_counter()
    frames, fps = load_video_rgb(video_path, max_frames=max_frames)
    t1 = time.perf_counter()
    mouths = crop_mouths(frames, auto_avsr_dir, face_model_path)
    t2 = time.perf_counter()
    x = preprocess_mouths(mouths, auto_avsr_dir).to(device)
    feats = encoder.to(device).eval()(x).float().cpu().numpy().astype(np.float16)
    t3 = time.perf_counter()
    info = {
        "n_frames": float(frames.shape[0]),
        "fps": fps,
        "sec_decode": t1 - t0,
        "sec_crop": t2 - t1,
        "sec_encode": t3 - t2,
    }
    return feats, info


# --------------------------------------------------------------------------------------
# Frame-rate normalization (both model families are configured for 25 fps input)
# --------------------------------------------------------------------------------------


def resample_to_fps(
    frames: np.ndarray, fps: float, target_fps: float = 25.0
) -> np.ndarray:
    """Pick frames on an even grid so the clip plays at ``target_fps``.

    Same rule as the multilingual VSR repo (``torch.linspace`` index selection with
    ``speed_rate = fps / target_fps``), applied to raw frames before cropping.
    """
    speed_rate = fps / target_fps
    if abs(speed_rate - 1.0) < 1e-3:
        return frames
    n_out = int(frames.shape[0] / speed_rate)
    idx = torch.linspace(0, frames.shape[0] - 1, n_out, dtype=torch.int64).numpy()
    return frames[idx]


# --------------------------------------------------------------------------------------
# Multilingual VSR models (Ma et al., 2022; repo mpc001/Visual_Speech_Recognition_for_
# Multiple_Languages). This repo also ships a top-level package named ``espnet`` that
# differs from Auto-AVSR's, so imports are done in an isolated module namespace.
# --------------------------------------------------------------------------------------

_REPO_PACKAGES = ("espnet", "datamodule", "preparation", "pipelines")


def switch_repo(repo_dir: str | Path) -> None:
    """Purge cached repo packages from ``sys.modules`` and put ``repo_dir`` first on the path.

    Encoders that were already built keep working: their classes stay referenced.
    """
    for name in list(sys.modules):
        if name.split(".")[0] in _REPO_PACKAGES:
            del sys.modules[name]
    p = str(Path(repo_dir).resolve())
    sys.path[:] = [x for x in sys.path if x != p]
    sys.path.insert(0, p)


class MultilingualVSREncoder(torch.nn.Module):
    """Encoder of a model from the multilingual VSR model zoo, built from its model.json."""

    def __init__(
        self, repo_dir: str | Path, model_conf: str | Path | None = None, args=None
    ) -> None:
        super().__init__()
        import argparse
        import json

        switch_repo(repo_dir)
        from espnet.nets.pytorch_backend.e2e_asr_transformer import E2E

        if args is None:
            with open(model_conf, "rb") as f:
                confs = json.load(f)
            args = argparse.Namespace(
                **(confs if isinstance(confs, dict) else confs[2])
            )
            labels_type = getattr(args, "labels_type", "char")
            if labels_type == "char":
                odim = len(args.char_list)
            else:  # unigram5000: <blank> + 5000 units + <eos>, as in pipelines/model.py
                tok = Path(repo_dir) / "pipelines" / "tokens" / "unigram5000_units.txt"
                odim = len(tok.read_text().splitlines()) + 2
        else:
            odim = 10  # test path only
        self.args = args
        self.model = E2E(odim, args)

    def load_checkpoint(self, ckpt_path: str | Path) -> dict[str, list[str]]:
        """Load weights non-strictly and report key mismatches."""
        state = torch.load(str(ckpt_path), map_location="cpu")
        result = self.model.load_state_dict(state, strict=False)
        return {
            "missing": list(result.missing_keys),
            "unexpected": list(result.unexpected_keys),
        }

    @torch.inference_mode()
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Map (1, T, 88, 88) normalized gray mouth frames to (T, D) features."""
        self.model.eval()
        return self.model.encode(x)


def preprocess_mouths_gray(mouths_rgb: np.ndarray) -> torch.Tensor:
    """Multilingual-repo transform: gray, /255, center crop 88, normalize -> (1, T, 88, 88).

    The original converts to gray before the affine warp; we convert the warped RGB
    crop. Both are linear operations, so the difference is limited to interpolation.
    """
    import torchvision

    gray = np.stack([cv2.cvtColor(f, cv2.COLOR_RGB2GRAY) for f in mouths_rgb])
    x = torch.from_numpy(gray).float().unsqueeze(0) / 255.0  # (1, T, H, W)
    x = torchvision.transforms.CenterCrop(88)(x)
    return torchvision.transforms.Normalize(0.421, 0.165)(x)


def extract_features_v2(
    video_path: str | Path,
    encoder: torch.nn.Module,
    auto_avsr_dir: str | Path,
    face_model_path: str | Path,
    device: str = "cpu",
    max_frames: int | None = 300,
    target_fps: float = 25.0,
) -> tuple[np.ndarray, dict[str, float]]:
    """Video -> 25 fps -> mouth crops -> encoder features (T, D) float16, for either family."""
    t0 = time.perf_counter()
    frames, fps = load_video_rgb(video_path, max_frames=max_frames)
    frames = resample_to_fps(frames, fps, target_fps)
    t1 = time.perf_counter()
    switch_repo(auto_avsr_dir)  # cropper lives in Auto-AVSR
    mouths = crop_mouths(frames, auto_avsr_dir, face_model_path)
    t2 = time.perf_counter()
    if isinstance(encoder, MultilingualVSREncoder):
        x = preprocess_mouths_gray(mouths).to(device)
    else:
        x = preprocess_mouths(mouths, auto_avsr_dir).to(device)
    feats = encoder.to(device).eval()(x).float().cpu().numpy().astype(np.float16)
    t3 = time.perf_counter()
    info = {
        "fps_in": fps,
        "n_frames_25fps": float(frames.shape[0]),
        "sec_decode": t1 - t0,
        "sec_crop": t2 - t1,
        "sec_encode": t3 - t2,
    }
    return feats, info

In [ ]:
# Cell 5: MAVOS-DD file layout (no video download yet)
# If the dataset is gated, add a Hugging Face token as a Colab secret named HF_TOKEN
# (key icon in the left sidebar) and accept the dataset terms on its Hugging Face page.
import collections
from huggingface_hub import HfApi
REPO = "unibuc-cs/MAVOS-DD"
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:  # noqa: BLE001
    HF_TOKEN = None
files = HfApi().list_repo_files(REPO, repo_type="dataset", token=HF_TOKEN)
ext = collections.Counter(Path(f).suffix.lower() for f in files)
top = collections.Counter(f.split("/")[0] for f in files)
layout = {"n_files": len(files), "extensions": dict(ext.most_common(10)),
          "top_level": dict(top.most_common(30)),
          "non_video_files": [f for f in files if not f.lower().endswith(".mp4")][:50]}
print(json.dumps(layout, indent=2))
(OUT / "mavos_layout.json").write_text(json.dumps(layout, indent=2))

In [ ]:
# Cell 6: download metadata only (no .mp4) and count clips per language x generator x label x split
from huggingface_hub import snapshot_download
from datasets import load_from_disk
import pandas as pd
META_DIR = Path("/content/mavos_meta")
snapshot_download(REPO, repo_type="dataset", local_dir=META_DIR, token=HF_TOKEN,
                  ignore_patterns=["*.mp4", "*.MP4", "*.wav"])
candidates = sorted({p.parent for p in META_DIR.rglob("state.json")}) or [META_DIR]
print("load_from_disk candidates:", candidates)
meta = load_from_disk(str(candidates[0]))
df = meta.to_pandas()
print(df.shape); print(df.head(3).to_string())
counts = (df.groupby(["language", "generative_method", "label", "split"]).size()
            .rename("n").reset_index())
counts.to_csv(OUT / "mavos_counts.csv", index=False)
print(pd.crosstab(df["language"], df["generative_method"]).to_string())
print(df.groupby(["split", "open_set_model", "open_set_language"]).size().to_string())

In [ ]:
# Cell 7: pick 3 clips (English real, English fake, Spanish real) and download them
import numpy as np
rng = np.random.default_rng(0)
file_set = set(files)
def pick(lang, label):
    sub = df[(df["language"] == lang) & (df["label"] == label)]
    if sub.empty:
        return None
    for i in rng.permutation(len(sub))[:20]:
        vp = sub.iloc[i]["video_path"]
        if vp in file_set:
            return vp
    return None
chosen = {"en_real": pick("english", "real"), "en_fake": pick("english", "fake"),
          "es_real": pick("spanish", "real")}
print(chosen)
from huggingface_hub import hf_hub_download
local = {}
for k, vp in chosen.items():
    if vp is None:
        print("No match for", k, "(check language/label values printed in Cell 6)")
        continue
    local[k] = hf_hub_download(REPO, filename=vp, repo_type="dataset", local_dir=CLIPS, token=HF_TOKEN)
print(local)

In [ ]:
# Cell 8: English VSR checkpoint from the official Auto-AVSR model zoo (README, "Model zoo", LRS3).
# vsr_trlrs2lrs3vox2avsp_base.pth: 3291 h training data, 20.3% WER on LRS3 (as reported upstream).
import hashlib, subprocess
CKPT_EN = WEIGHTS / "vsr_trlrs2lrs3vox2avsp_base.pth"
GDRIVE_ID = "1r1kx7l9sWnDOCnaFHIGvOtzuhFyFA88_"
MD5_PREFIX = "49f77"   # from the model zoo table
for stale in [WEIGHTS / "vsr_trlrs3_base.pth", CKPT_EN]:   # remove empty files from failed downloads
    if stale.exists() and stale.stat().st_size < 10_000_000:
        print("removing incomplete file:", stale, stale.stat().st_size, "bytes"); stale.unlink()
if not CKPT_EN.exists():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
    import gdown
    gdown.download(id=GDRIVE_ID, output=str(CKPT_EN), quiet=False)
size_mb = CKPT_EN.stat().st_size / 1e6
md5 = hashlib.md5(CKPT_EN.read_bytes()).hexdigest()
print(f"size: {size_mb:.0f} MB  md5: {md5}")
assert md5.startswith(MD5_PREFIX), "MD5 mismatch: delete the file and download it manually (see markdown note)"
sys.path.insert(0, "/content")
import step0_helpers as h
enc_en = h.VSREncoder(AUTO_AVSR)
keys_en = enc_en.load_checkpoint(CKPT_EN)
print("missing:", len(keys_en["missing"]), "unexpected:", len(keys_en["unexpected"]))
print("first missing:", keys_en["missing"][:5]); print("first unexpected:", keys_en["unexpected"][:5])

In [ ]:
# Cell 9: extract features on GPU, record timings and peak memory
FEAT = OUT / "features_sample"; FEAT.mkdir(exist_ok=True)
torch.cuda.reset_peak_memory_stats()
clip_report = {}
for k, path in local.items():
    try:
        f, info = h.extract_features(path, enc_en, AUTO_AVSR, FACE_MODEL, device="cuda", max_frames=250)
        np.save(FEAT / f"{k}_en.npy", f)
        info.update({"shape": list(f.shape), "finite": bool(np.isfinite(f).all())})
        clip_report[k] = info
    except Exception as e:  # noqa: BLE001
        clip_report[k] = {"error": f"{type(e).__name__}: {e}"}
    print(k, clip_report[k])
peak_gb = torch.cuda.max_memory_allocated() / 1e9
print("peak GPU memory (GB):", round(peak_gb, 2))

## Optional: Spanish encoder (decides Study A)

Download the Spanish visual-only model from the model zoo table in
https://github.com/mpc001/Visual_Speech_Recognition_for_Multiple_Languages (README, Model Zoo, Spanish),
and save it to `MyDrive/lipshift/weights/` with its original file name. Then run the next cell.

In [ ]:
# Cell 10: does the Spanish checkpoint fit the same encoder?
es_ckpts = sorted(p for p in WEIGHTS.glob("*.pth") if p.name != CKPT_EN.name)
spanish = {"found": [p.name for p in es_ckpts]}
if es_ckpts:
    enc_es = h.VSREncoder(AUTO_AVSR)
    k = enc_es.load_checkpoint(es_ckpts[0])
    spanish.update({"file": es_ckpts[0].name, "missing": len(k["missing"]), "unexpected": len(k["unexpected"]),
                    "first_missing": k["missing"][:5], "first_unexpected": k["unexpected"][:5]})
    if "es_real" in local and spanish["missing"] == 0:
        f, info = h.extract_features(local["es_real"], enc_es, AUTO_AVSR, FACE_MODEL, device="cuda")
        spanish["es_real_shape"] = list(f.shape)
print(json.dumps(spanish, indent=2))

In [ ]:
# Cell 11: go / no-go summary
ok_clips = [k for k, v in clip_report.items() if "error" not in v]
secs = [v["sec_decode"] + v["sec_crop"] + v["sec_encode"] for v in clip_report.values() if "error" not in v]
vid_secs = [v["n_frames"] / v["fps"] for v in clip_report.values() if "error" not in v]
summary = {
    "env": report,
    "mavos_n_files": layout["n_files"],
    "mavos_rows": int(len(df)),
    "en_ckpt_missing_keys": len(keys_en["missing"]),
    "clips": clip_report,
    "peak_gpu_gb": round(peak_gb, 2),
    "processing_sec_per_video_sec": round(sum(secs) / sum(vid_secs), 3) if vid_secs else None,
    "spanish": spanish,
}
summary["GO_core"] = bool(len(keys_en["missing"]) == 0 and len(ok_clips) >= 2)
summary["GO_study_A"] = bool(spanish.get("missing", -1) == 0)
(OUT / "step0_report.json").write_text(json.dumps(summary, indent=2, default=str))
print(json.dumps(summary, indent=2, default=str))